In [1]:
import os
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from anndata import AnnData
from tqdm.notebook import tqdm
import scanpy.external as sce
import scipy

import warnings 
warnings.filterwarnings('ignore')

In [2]:
adata_4mAD = ad.read('4mAD_mousebrain.h5ad')
adata_4mWT = ad.read('4mWT_mousebrain.h5ad')
adata_8mAD = ad.read('8mAD_mousebrain.h5ad')
adata_8mWT = ad.read('8mWT_mousebrain.h5ad')
adata_14mWT = ad.read('14mWT_mousebrain.h5ad')
adata_14mAD = ad.read('14mAD_mousebrain.h5ad')

In [3]:
samples = [
    {
        "name": "4mAD",
        "file": "4mAD_mousebrain.h5ad",
        "group": "AD",
        "age": "4m",
        "type": "4mAD"
    },
    {
        "name": "4mWT",
        "file": "4mWT_mousebrain.h5ad",
        "group": "WT",
        "age": "4m",
        "type": "4mWT"
    },
    {
        "name": "8mAD",
        "file": "8mAD_mousebrain.h5ad",
        "group": "AD",
        "age": "8m",
        "type": "8mAD"
    },
    {
        "name": "8mWT",
        "file": "8mWT_mousebrain.h5ad",
        "group": "WT",
        "age": "8m",
        "type": "8mWT"
    },
    {
        "name": "14mWT",
        "file": "14mWT_mousebrain.h5ad",
        "group": "WT",
        "age": "14m",
        "type": "14mWT"
    },
    {
        "name": "14mAD",
        "file": "14mAD_mousebrain.h5ad",
        "group": "AD",
        "age": "14m",
        "type": "14mAD"
    }
]
all_adatas = []
all_remain_reads_info = []

for sample in samples:
    adata = ad.read(sample["file"])
    prefix = sample["name"]
    adata.obs = adata.obs.copy()
    adata.obs["original_barcode"] = adata.obs.index.astype(str)
    adata.obs.index = prefix + "-" + adata.obs.index.astype(str)
    adata.obs["cell_barcode"] = adata.obs.index.astype(str)
    adata.obs["group"] = sample["group"]
    adata.obs["age"] = sample["age"]
    adata.obs["type"] = sample["type"]

    # remain_df = None
    # if "remain_reads_info" in adata.uns:
    #     remain_df = adata.uns["remain_reads_info"].copy()
    #     remain_df["original_barcode"] = remain_df["cell_barcode"].astype(str)
    #     remain_df["cell_barcode"] = prefix + "-" + remain_df["cell_barcode"].astype(str)
    #     remain_df["group"] = sample["group"]
    #     remain_df["age"] = sample["age"]
    #     remain_df["type"] = sample["type"]
    #     all_remain_reads_info.append(remain_df)

    all_adatas.append(adata)

adata_merged = ad.concat(all_adatas, axis=0, join="outer", label="batch", fill_value=0, index_unique=None)

# if all_remain_reads_info:
#     remain_reads_info_merged = pd.concat(all_remain_reads_info, axis=0, ignore_index=True)
#     adata_merged.uns["remain_reads_info"] = remain_reads_info_merged


In [4]:
adata_merged.obs

,column,row,z,original_barcode,cell_barcode,group,age,type,batch
cell_barcode,,,,,,,,,
4mAD-38,2502,5783,8,38,4mAD-38,AD,4m,4mAD,0
4mAD-408,2500,5779,6,408,4mAD-408,AD,4m,4mAD,0
4mAD-411,2494,5782,6,411,4mAD-411,AD,4m,4mAD,0
4mAD-418,2497,5785,9,418,4mAD-418,AD,4m,4mAD,0
4mAD-478,2395,5812,12,478,4mAD-478,AD,4m,4mAD,0
...,...,...,...,...,...,...,...,...,...
14mAD-385010,32133,30670,24,385010,14mAD-385010,AD,14m,14mAD,5
14mAD-385011,32298,30724,27,385011,14mAD-385011,AD,14m,14mAD,5
14mAD-385014,31953,30769,9,385014,14mAD-385014,AD,14m,14mAD,5


In [6]:
adata_merged.var

""
1700019D03Rik
A2m
AW551984
Aamp
Abat
...
Znrf1
Zranb2
Zzz3
mt-Co1


filtering

In [7]:
import numpy as np
#import matplotlib.pyplot as plt  

filtered_cell_barcodes = []

adatas_by_type = []
for t in adata_merged.obs["type"].unique():
    print(f"\n--- Filtering type: {t} ---")
    ad_t = adata_merged[adata_merged.obs["type"] == t].copy()
    n_init_cells = ad_t.n_obs
    n_init_genes = ad_t.n_vars

    # --------- cell ---------
    # 1. 10 genes
    n_genes_by_cell = (ad_t.X > 0).sum(axis=1)
    if hasattr(n_genes_by_cell, "A1"):  # sparse matrix
        n_genes_by_cell = n_genes_by_cell.A1
    elif hasattr(n_genes_by_cell, "toarray"):
        n_genes_by_cell = np.asarray(n_genes_by_cell).ravel()
    else:
        n_genes_by_cell = np.array(n_genes_by_cell).ravel()

    #plt.figure(figsize=(5,2))
    #plt.hist(n_genes_by_cell, bins=50)
    #plt.xlabel("Num genes expressed per cell")
    #plt.ylabel("Num cells")
    #plt.title(f"{t}: #genes/cell before gene count filter")
    #plt.tight_layout()
    #plt.show()

    cell_mask_1 = n_genes_by_cell >= 10
    print(f"  Cells removed by <10 gene criterion: {np.sum(~cell_mask_1)} / {n_init_cells}")

    # 2. ±3*mad
    if "total_counts" in ad_t.obs:
        total_counts = ad_t.obs["total_counts"].values
    else:
        total_counts = ad_t.X.sum(axis=1)
        if hasattr(total_counts, "A1"):
            total_counts = total_counts.A1
        elif hasattr(total_counts, "toarray"):
            total_counts = np.asarray(total_counts).ravel()
        else:
            total_counts = np.array(total_counts).ravel()

    #plt.figure(figsize=(5,2))
    #plt.hist(total_counts, bins=50)
    #plt.xlabel("Total counts per cell")
    #plt.ylabel("Num cells")
    #plt.title(f"{t}: total counts/cell before MAD filter")
    #plt.tight_layout()
    #plt.show()

    median = np.median(total_counts)
    mad = np.median(np.abs(total_counts - median))
    cell_mask_2 = (total_counts >= median - 3*mad) & (total_counts <= median + 3*mad)
    print(f"  Cells removed by 3MAD total_count: {np.sum(~cell_mask_2)} / {n_init_cells}")

    cell_mask = cell_mask_1 & cell_mask_2
    print(f"  Cells retained after both filters: {np.sum(cell_mask)} / {n_init_cells} ({np.sum(cell_mask)/n_init_cells:.2%})")

    retained_barcodes = ad_t.obs.index[cell_mask]
    filtered_cell_barcodes.extend(retained_barcodes.tolist())

    ad_t = ad_t[retained_barcodes].copy()

    # --------- gene ---------
    n_cells_by_gene = (ad_t.X > 0).sum(axis=0)
    if hasattr(n_cells_by_gene, "A1"):  # sparse matrix
        n_cells_by_gene = n_cells_by_gene.A1
    elif hasattr(n_cells_by_gene, "toarray"):
        n_cells_by_gene = np.asarray(n_cells_by_gene).ravel()
    else:
        n_cells_by_gene = np.array(n_cells_by_gene).ravel()

    #plt.figure(figsize=(5,2))
    #plt.hist(n_cells_by_gene, bins=50)
    #plt.xlabel("Num cells w/gene >0")
    #plt.ylabel("Num genes")
    #plt.title(f"{t}: #cells/gene before gene filter")
    #plt.tight_layout()
    #plt.show()

    gene_mask = n_cells_by_gene >= 10
    n_removed_genes = np.sum(~gene_mask)
    print(f"  Genes removed (<10 cells): {n_removed_genes} / {n_init_genes}")

    ad_t = ad_t[:, ad_t.var.index[gene_mask]].copy()

    print(f"  After filtering: {ad_t.n_obs} cells, {ad_t.n_vars} genes.")

    adatas_by_type.append(ad_t)

adata_filtered = ad.concat(adatas_by_type, axis=0)
print("\n==> Filtered adata shape:", adata_filtered.shape)


if "remain_reads_info" in adata_merged.uns:
    remain_reads_info = adata_merged.uns["remain_reads_info"]
    mask_nuclei_minus1 = remain_reads_info["nuclei"] == -1
    mask_barcode_in_list = remain_reads_info["cell_barcode"].isin(filtered_cell_barcodes)
    remain_reads_info_filtered = remain_reads_info[mask_nuclei_minus1 | mask_barcode_in_list].copy()
    adata_filtered.uns["remain_reads_info"] = remain_reads_info_filtered

adata_filtered


--- Filtering type: 4mAD ---
  Cells removed by <10 gene criterion: 450 / 27819
  Cells removed by 3MAD total_count: 1387 / 27819
  Cells retained after both filters: 25982 / 27819 (93.40%)
  Genes removed (<10 cells): 0 / 3019
  After filtering: 25982 cells, 3019 genes.

--- Filtering type: 4mWT ---
  Cells removed by <10 gene criterion: 606 / 36365
  Cells removed by 3MAD total_count: 1827 / 36365
  Cells retained after both filters: 33932 / 36365 (93.31%)
  Genes removed (<10 cells): 0 / 3019
  After filtering: 33932 cells, 3019 genes.

--- Filtering type: 8mAD ---
  Cells removed by <10 gene criterion: 201 / 32530
  Cells removed by 3MAD total_count: 1932 / 32530
  Cells retained after both filters: 30397 / 32530 (93.44%)
  Genes removed (<10 cells): 0 / 3019
  After filtering: 30397 cells, 3019 genes.

--- Filtering type: 8mWT ---
  Cells removed by <10 gene criterion: 530 / 27241
  Cells removed by 3MAD total_count: 1561 / 27241
  Cells retained after both filters: 25150 / 27241

AnnData object with n_obs × n_vars = 169262 × 3019
    obs: 'column', 'row', 'z', 'original_barcode', 'cell_barcode', 'group', 'age', 'type', 'batch'
    layers: 'ntRNA', 'rbRNA'

In [9]:
# any type,max_counts > 3
max_counts_per_type = (
    adata_filtered.to_df()
    .groupby(adata_filtered.obs['type'])
    .max()
)
keep_gene_mask = (max_counts_per_type > 3).any(axis=0)
filtered_genes = max_counts_per_type.columns[keep_gene_mask]
removed_genes = max_counts_per_type.columns[~keep_gene_mask]

print(f"{keep_gene_mask.sum()} / {len(keep_gene_mask)}")

if len(removed_genes) > 0:
    print("removed genes:")
    for g in removed_genes:
        print(g)

adata_filtered = adata_filtered[:, filtered_genes].copy()
adata_filtered

2842 / 3019
removed genes:
AW551984
Acbd7
Acsl1
Afp
Angpt1
Arg1
Arnt
Asic4
Atf1
Bmp4
Calcr
Cblb
Cckar
Ccl7
Cd300c2
Cd44
Chd6
Clpp
Col20a1
Col25a1
Col26a1
Cpxm2
Crct1
Crisp1
Cxcl2
Cyp2s1
Dclk3
Deaf1
Dhh
Dlx6
Drd1
Ebf3
Efr3a
Eif1a
Erlin1
Erlin2
Etfa
Ets1
Fam169b
Fbxo41
Fip1l1
Foxo1
Fst
Gapdh
Gar1
Gbx1
Gfm1
Ghrh
Gna14
Hcrtr2
Hdhd3
Helz
Higd1b
Hoxa10
Hoxa5
Hoxa7
Hoxa9
Hoxb6
Hoxb7
Hoxc9
Hr
Hrh4
Hs3st2
Hspa8
Htr1a
Htr1f
Htr2b
Igfbpl1
Ikzf1
Il1rapl2
Il23a
Isl1
Kcnc1
Keg1
Klf15
Krt27
Lancl3
Maml1
Me1
Mfap1b
Mgst1
Mkx
Mlxip
Ms4a15
Nfatc1
Nfe2l3
Ngb
Ngfr
Nmbr
Nmur2
Nog
Notum
Npbwr1
Nppa
Npy1r
Npy2r
Nr2c1
Nr2e1
Ntf3
Nxph2
Pbx4
Pcdh15
Pdap1
Pds5b
Pdyn
Per2
Pik3cd
Pitx2
Pla2g5
Pou3f4
Pparg
Ppil4
Ppp1r17
Procr
Prr7
Ptpn23
Ptpro
Rfx1
Robo2
Rpl24
Rpl34
Rpsa
Rxfp3
Sall3
Samsn1
Sapcd2
Satb1
Scgn
Scn10a
Scn4b
Scn9a
Scrn3
Sec61a1
Sema3a
Sema3c
Sema5a
Septin4
Serpinb1a
Serpinb1b
Shisa8
Shox2
Siglech
Sla2
Sla
Slc13a3
Slc17a6
Slc26a3
Slc6a2
Sntb1
Snx33
Socs3
Sox6
Spon2
St8sia4
Syt2
Tbk1
Tbx18
Tfam
Tmbim1
Tmc

AnnData object with n_obs × n_vars = 169262 × 2842
    obs: 'column', 'row', 'z', 'original_barcode', 'cell_barcode', 'group', 'age', 'type', 'batch'
    layers: 'ntRNA', 'rbRNA'

In [10]:
adata_filtered

AnnData object with n_obs × n_vars = 169262 × 2842
    obs: 'column', 'row', 'z', 'original_barcode', 'cell_barcode', 'group', 'age', 'type', 'batch'
    layers: 'ntRNA', 'rbRNA'

In [11]:
import numpy as np

adata = adata_filtered

adata.layers["totalRNA_raw"] = adata.X.copy()

adata.layers["ntRNA_raw"] = adata.layers.pop("ntRNA")
adata.layers["rbRNA_raw"] = adata.layers.pop("rbRNA")

adata.layers["TE"] = adata.layers["rbRNA_raw"] / (adata.layers["totalRNA_raw"])

In [12]:
adata

AnnData object with n_obs × n_vars = 169262 × 2842
    obs: 'column', 'row', 'z', 'original_barcode', 'cell_barcode', 'group', 'age', 'type', 'batch'
    layers: 'totalRNA_raw', 'ntRNA_raw', 'rbRNA_raw', 'TE'

In [13]:
import numpy as np
import scipy.sparse as sp

adata = adata_filtered

def cell_sum(X):
    if sp.issparse(X):
        return np.asarray(X.sum(axis=1)).ravel()
    else:
        return X.sum(axis=1)

total_counts = cell_sum(adata.layers["totalRNA_raw"])

median_counts = np.median(total_counts)
print(f"Median totalRNA_raw counts per cell: {median_counts:.2f}")

# scaling factor = median / cell_total
scale_factor = median_counts / (total_counts + 1e-8)

if sp.issparse(adata.layers["totalRNA_raw"]):
    totalRNA_norm = adata.layers["totalRNA_raw"].multiply(scale_factor[:, None])
else:
    totalRNA_norm = adata.layers["totalRNA_raw"] * scale_factor[:, None]

adata.layers["totalRNA_norm"] = totalRNA_norm

adata.layers["totalRNA_log"] = (
    totalRNA_norm.log1p() if sp.issparse(totalRNA_norm)
    else np.log1p(totalRNA_norm)
)

adata.layers["rbRNA_norm"] = (
    adata.layers["totalRNA_norm"].multiply(adata.layers["TE"])
    if sp.issparse(adata.layers["totalRNA_norm"])
    else adata.layers["totalRNA_norm"] * adata.layers["TE"]
)

rbRNA_norm = adata.layers["rbRNA_norm"]
adata.layers["rbRNA_log"] = (
    rbRNA_norm.log1p() if sp.issparse(rbRNA_norm)
    else np.log1p(rbRNA_norm)
)

Median totalRNA_raw counts per cell: 286.00


In [14]:
import scipy.sparse as sp

for k in adata.layers.keys():
    if sp.issparse(adata.layers[k]):
        adata.layers[k] = adata.layers[k].tocsr()

In [15]:
adata

AnnData object with n_obs × n_vars = 169262 × 2842
    obs: 'column', 'row', 'z', 'original_barcode', 'cell_barcode', 'group', 'age', 'type', 'batch'
    layers: 'totalRNA_raw', 'ntRNA_raw', 'rbRNA_raw', 'TE', 'totalRNA_norm', 'totalRNA_log', 'rbRNA_norm', 'rbRNA_log'

In [16]:
adata.write('ADcombined_mousebrain.h5ad')